# 06 — Subclassing `EdrDataStore`

`EdrDataStore` is the orchestrator behind `xr.open_dataset(...,
engine="edr")`. It exposes **seven hook methods** — every one of them
is a documented extension point that downstream packages (and
end-users) can override.

| Hook | Purpose |
|---|---|
| `_request` | low-level HTTP — auth, signing, retry, logging |
| `_parse_collection_metadata` | server-specific metadata extensions |
| `_negotiate_output_format` | prefer NetCDF over CoverageJSON, etc. |
| `_build_cube_url` | non-standard URL routing (e.g. `/data` instead of `/cube`) |
| `_parse_coveragejson` | server-specific CoverageJSON quirks, response caching |
| `_translate_indexer` | custom slicing semantics |
| `_discover_axes` | custom axis discovery (e.g. service-specific catalog) |

This notebook gives a working example for each, then shows how to use
a subclass directly (without re-registering the xarray engine).

In [ ]:
# Replace with your EDR collection URL
collection_url = "https://edr.example.com/collections/temperature_2m"

## Hook 1: `_request` — log every HTTP call

Useful for debugging request URLs and headers in development.

In [ ]:
class LoggingStore(EdrDataStore):
    def _request(self, method, url, *, params=None, headers=None):
        print(f"[http] {method} {url} params={dict(params or {})}")
        return super()._request(method, url, params=params, headers=headers)


store = LoggingStore(collection_url=collection_url)
ds = store.build_dataset()
_ = ds["temperature"].values
ds.close()

## Hook 2: `_parse_collection_metadata` — handle server extensions

If your server adds non-standard fields (vendor extensions, alternate
parameter formats), parse them here. The default implementation calls
`metadata.parse_collection_metadata`.

In [ ]:
class TaggedMetadataStore(EdrDataStore):
    def _parse_collection_metadata(self, payload):
        # Inspect or mutate the raw dict before delegating.
        custom_tag = payload.get("x-vendor-tag")
        result = super()._parse_collection_metadata(payload)
        print(f"[metadata] vendor tag: {custom_tag!r}, parsed id: {result.id}")
        return result


store = TaggedMetadataStore(collection_url=collection_url)
ds = store.build_dataset()
ds.close()

## Hook 3: `_negotiate_output_format` — pick a different format

The default selects `CoverageJSON` if the collection advertises it.
Override to pick something else (NetCDF, GeoTIFF) when your server
supports it. Here we just confirm the default behavior.

In [ ]:
class StrictFormatStore(EdrDataStore):
    def _negotiate_output_format(self, advertised):
        # In production: prefer NetCDF when available, else CoverageJSON.
        chosen = super()._negotiate_output_format(advertised)
        print(f"[format] advertised={advertised} chose={chosen}")
        return chosen


store = StrictFormatStore(collection_url=collection_url)
ds = store.build_dataset()
ds.close()

## Hook 4: `_build_cube_url` — non-standard routing

Some servers expose the cube under `/data` or `/query` rather than the
canonical `/cube` path. Override this hook if `cube_link.href` does not
follow the OGC EDR convention.

In [ ]:
class CustomUrlStore(EdrDataStore):
    def _build_cube_url(self, collection_url, instance):
        # Default delegates to metadata.cube_url. Here we just log
        # the result for visibility.
        url = super()._build_cube_url(collection_url, instance)
        print(f"[url] cube endpoint: {url}")
        return url


store = CustomUrlStore(collection_url=collection_url)
ds = store.build_dataset()
ds.close()

## Hook 5: `_parse_coveragejson` — cache responses

Wrap the CoverageJSON parser to cache parsed results in memory. Useful
when many DataArrays might be derived from the same fetch.

In [ ]:
class CachingStore(EdrDataStore):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._coverage_cache: dict[int, object] = {}

    def _parse_coveragejson(self, payload):
        key = id(payload)
        if key in self._coverage_cache:
            print("[cache] hit")
            return self._coverage_cache[key]
        result = super()._parse_coveragejson(payload)
        self._coverage_cache[key] = result
        print("[cache] miss; parsed and cached")
        return result


store = CachingStore(collection_url=collection_url)
ds = store.build_dataset()
_ = ds["temperature"].values
ds.close()

## Hook 6: `_translate_indexer` — custom slicing

The default translator turns xarray's `BasicIndexer` key tuple into
EDR query parameters (`bbox`, `datetime`, `z`). Override if your server
uses different slicing conventions (e.g. integer indices instead of
geographic coordinates).

In [ ]:
class VerboseIndexStore(EdrDataStore):
    def _translate_indexer(self, key, axes):
        params = super()._translate_indexer(key, axes)
        print(f"[indexer] key={key} -> params={params}")
        return params


store = VerboseIndexStore(collection_url=collection_url)
ds = store.build_dataset()
_ = ds["temperature"].isel(t=0).values
ds.close()

## Hook 7: `_discover_axes` — custom axis discovery

Override when neither `probe`, `metadata_only`, nor `strict` matches
how your server advertises grid axes (e.g. an out-of-band catalog
service).

In [ ]:
class CatalogDiscoveryStore(EdrDataStore):
    def _discover_axes(self, metadata):
        # Default: dispatch on self.discovery (probe / metadata_only / strict).
        # Override entirely to consult your own catalog. Here we just log.
        axes = super()._discover_axes(metadata)
        print(f"[discovery] axes: {[a.name for a in axes]}")
        return axes


store = CatalogDiscoveryStore(collection_url=collection_url)
ds = store.build_dataset()
ds.close()

## Using a subclass directly

You don't need to register a new xarray entry point to use a subclass:
just instantiate it and call `build_dataset()`. The returned dataset
is identical to what `xr.open_dataset(..., engine="edr")` would
produce.

In [ ]:
class CombinedStore(LoggingStore, CachingStore):
    """Mix multiple hooks via cooperative MRO."""


store = CombinedStore(
    collection_url=collection_url,
    parameter_names=["temperature"],
)
ds = store.build_dataset()
print()
print("dims:    ", dict(ds.dims))
print("vars:    ", list(ds.data_vars))
ds.close()